# 03 — Model Training, Tuning and Evaluation

## Metric rationale
**Average Precision (PR-AUC)** is the primary model-selection metric because it summarizes precision-recall performance under class imbalance. **Recall** is important because false negatives represent missed at-risk accounts. Precision indicates the Customer Success review workload; F1 balances precision and recall. ROC-AUC is reported as a complementary ranking measure.

All results are illustrative because the dataset and labels are synthetic.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split,StratifiedKFold,GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier,HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score,roc_auc_score,precision_score,recall_score,f1_score,classification_report,ConfusionMatrixDisplay,PrecisionRecallDisplay
from sklearn.inspection import permutation_importance

ROOT=Path.cwd()
if ROOT.name=='notebooks': ROOT=ROOT.parent
IMG=ROOT/'images'; IMG.mkdir(exist_ok=True)
df=pd.read_csv(ROOT/'data/raw/enterprise_saas_churn_synthetic.csv')
X=df.drop(columns=['churned','account_id']); y=df.churned.astype(int)
cat=X.select_dtypes(include='object').columns.tolist(); num=X.select_dtypes(exclude='object').columns.tolist()
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.20,stratify=y,random_state=42)
prep=ColumnTransformer([('num',Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num),('cat',Pipeline([('impute',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]),cat)])
cv=StratifiedKFold(n_splits=3,shuffle=True,random_state=42)
print('Training rows:',len(X_train),'Test rows:',len(X_test),'Churn prevalence:',round(y.mean(),3))

## 1. Compare classifiers using GridSearchCV

All models use the same preprocessing pipeline. GridSearchCV tunes on the training partition only and ranks models by mean cross-validated Average Precision.

In [ ]:
models={
 'Logistic Regression':(LogisticRegression(max_iter=2000,class_weight='balanced',random_state=42),{'model__C':[0.1,1.0]}),
 'Decision Tree':(DecisionTreeClassifier(class_weight='balanced',random_state=42),{'model__max_depth':[3,5,None],'model__min_samples_leaf':[10,25]}),
 'Random Forest':(RandomForestClassifier(class_weight='balanced',random_state=42,n_jobs=1),{'model__n_estimators':[80],'model__max_depth':[6,None],'model__min_samples_leaf':[10]}),
 'Hist Gradient Boosting':(HistGradientBoostingClassifier(random_state=42),{'model__learning_rate':[0.05,0.1],'model__max_iter':[80],'model__max_leaf_nodes':[15]})}
searches={}; cv_rows=[]
for name,(est,grid) in models.items():
    pipe=Pipeline([('preprocess',prep),('model',est)])
    search=GridSearchCV(pipe,grid,scoring='average_precision',cv=cv,n_jobs=1,refit=True)
    search.fit(X_train,y_train); searches[name]=search
    cv_rows.append({'model':name,'cv_average_precision':search.best_score_,'best_params':str(search.best_params_)})
cv_results=pd.DataFrame(cv_rows).sort_values('cv_average_precision',ascending=False)
display(cv_results)

## 2. Holdout metrics and selected model

The model with the highest cross-validated Average Precision is selected. The holdout set is then evaluated using the same 0.50 classification threshold for a consistent baseline comparison. Threshold optimization should be performed on validation data if this were a production use case.

In [ ]:
rows=[]
for name,search in searches.items():
    model=search.best_estimator_; pred=model.predict(X_test); score=model.predict_proba(X_test)[:,1]
    rows.append({'model':name,'precision':precision_score(y_test,pred,zero_division=0),'recall':recall_score(y_test,pred,zero_division=0),'f1':f1_score(y_test,pred,zero_division=0),'average_precision':average_precision_score(y_test,score),'roc_auc':roc_auc_score(y_test,score)})
test_results=pd.DataFrame(rows).sort_values('average_precision',ascending=False)
display(test_results.style.format({c:'{:.3f}' for c in test_results.columns if c!='model'}))
best_name=cv_results.iloc[0]['model']; best=searches[best_name].best_estimator_
pred=best.predict(X_test); score=best.predict_proba(X_test)[:,1]
print('Selected model:',best_name)
print(classification_report(y_test,pred,target_names=['Retained','Churned'],zero_division=0))
fig,axes=plt.subplots(1,2,figsize=(12,4))
ConfusionMatrixDisplay.from_predictions(y_test,pred,display_labels=['Retained','Churned'],ax=axes[0],colorbar=False)
axes[0].set_title('Holdout confusion matrix')
PrecisionRecallDisplay.from_predictions(y_test,score,ax=axes[1],name=best_name)
axes[1].set_title('Holdout precision-recall curve')
plt.tight_layout(); plt.savefig(IMG/'model_confusion_and_pr_curve.png',dpi=160); plt.show()
fig,ax=plt.subplots(figsize=(10,5))
plot=test_results.melt(id_vars='model',value_vars=['precision','recall','f1','average_precision','roc_auc'],var_name='metric',value_name='score')
sns.barplot(data=plot,x='model',y='score',hue='metric',palette='deep',ax=ax)
ax.set(title='Holdout model comparison',xlabel='Classifier',ylabel='Score',ylim=(0,1)); ax.tick_params(axis='x',rotation=15)
plt.tight_layout(); plt.savefig(IMG/'model_comparison.png',dpi=160); plt.show()

## 3. Global model interpretation

Permutation importance measures the change in holdout Average Precision after shuffling one feature. It is a global diagnostic, not a causal explanation. Correlated predictors can share or mask importance.

In [ ]:
perm=permutation_importance(best,X_test,y_test,scoring='average_precision',n_repeats=5,random_state=42,n_jobs=1)
importance=pd.DataFrame({'feature':X.columns,'importance_mean':perm.importances_mean,'importance_std':perm.importances_std}).sort_values('importance_mean',ascending=False)
display(importance.head(12))
fig,ax=plt.subplots(figsize=(9,5))
sns.barplot(data=importance.head(12),x='importance_mean',y='feature',color='slateblue',ax=ax)
ax.set(title='Top permutation importances',xlabel='Decrease in Average Precision after shuffling',ylabel='Feature')
plt.tight_layout(); plt.savefig(IMG/'feature_importance.png',dpi=160); plt.show()
(ROOT/'data/processed').mkdir(exist_ok=True)
cv_results.to_csv(ROOT/'data/processed/cross_validation_results.csv',index=False)
test_results.to_csv(ROOT/'data/processed/holdout_model_metrics.csv',index=False)
importance.to_csv(ROOT/'data/processed/permutation_importance.csv',index=False)

## 4. Findings and business recommendations

In this simulation, the selected Logistic Regression model achieved holdout Average Precision of approximately 0.650 and recall of approximately 0.837. Its precision was approximately 0.438, indicating that a substantial share of flagged accounts would not be churn-positive under the simulated labels. Random Forest had higher precision and F1 in this run, while Logistic Regression provided higher recall and Average Precision.

Use adoption, engagement, satisfaction, value-realization and support trends as prompts for structured account reviews—not as automatic churn decisions. Validate thresholds on governed, time-stamped customer data before operational use.

## 5. Limitations and next steps

- The dataset and target are synthetic; model metrics are not evidence of production readiness.
- A random split demonstrates the workflow but does not simulate forecasting future renewal outcomes.
- A production study needs real historical account data, clear observation and churn windows, leakage controls, temporal validation, calibration, segment-level evaluation and human review.
- Run a controlled pilot and measure incremental retained ARR against intervention cost before scaling.